# 🏋️ Challenge 06 — Object-Oriented Programming
*Covers Modules 13–14.* 🟢 easy · 🟡 medium · 🔴 hard.

In [1]:
def check(name, test):
    try:
        ok = bool(test())
    except Exception as e:
        ok = False
        name += f"   (raised {type(e).__name__}: {e})"
    print("✅ PASS" if ok else "❌ FAIL", "-", name)

def raises(exc, fn, *a, **k):
    """Helper for tests: True if fn(*a, **k) raises exc."""
    try:
        fn(*a, **k)
    except exc:
        return True
    except Exception:
        return False
    return False

### 🟢 Problem 1 — A `Rectangle` class
Attributes `width`, `height`. Methods `area()`, `perimeter()`, `is_square()`. Add `__str__` returning `"Rectangle(3x4)"` and `__eq__` (equal if same width and height).

In [2]:
class Rectangle:
    def __init__(self, width, height):
        self.width, self.height = width, height

    def area(self): return self.width * self.height
    def perimeter(self): return 2 * (self.width + self.height)
    def is_square(self): return self.width == self.height
    def __str__(self): return f"Rectangle({self.width}x{self.height})"
    def __eq__(self, other):
        return isinstance(other, Rectangle) and (self.width, self.height) == (other.width, other.height)

In [3]:
r = Rectangle(3, 4)
check("area", lambda: r.area() == 12)
check("perimeter", lambda: r.perimeter() == 14)
check("is_square", lambda: Rectangle(2, 2).is_square() and not r.is_square())
check("__str__", lambda: str(r) == "Rectangle(3x4)")
check("__eq__", lambda: Rectangle(3, 4) == r and r != Rectangle(4, 3))

✅ PASS - area
✅ PASS - perimeter
✅ PASS - is_square
✅ PASS - __str__
✅ PASS - __eq__


### 🟢 Problem 2 — `Counter` with class attribute
Class `Student` with instance attrs `name`, `marks` (list) and a **class attribute** `count` tracking how many students were created. Methods: `average()` (0 if no marks), `add_mark(m)`.

In [4]:
class Student:
    count = 0

    def __init__(self, name):
        self.name = name
        self.marks = []
        Student.count += 1

    def add_mark(self, m):
        self.marks.append(m)

    def average(self):
        return sum(self.marks) / len(self.marks) if self.marks else 0

In [5]:
Student.count = 0
a, b = Student("Asha"), Student("Ravi")
a.add_mark(80); a.add_mark(90)
check("count tracks instances", lambda: Student.count == 2)
check("average", lambda: a.average() == 85)
check("average with no marks", lambda: b.average() == 0)
check("marks are per-instance", lambda: b.marks == [] and a.marks == [80, 90])

✅ PASS - count tracks instances
✅ PASS - average
✅ PASS - average with no marks
✅ PASS - marks are per-instance


### 🟢 Problem 3 — Encapsulation with `@property`
`Temperature` stores Celsius privately. Property `celsius` (with validation: below −273.15 raises `ValueError`), read-only property `fahrenheit`, and a settable property `kelvin`.

In [6]:
class Temperature:
    def __init__(self, celsius=0):
        self.celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("below absolute zero")
        self._celsius = value

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32

    @property
    def kelvin(self):
        return self._celsius + 273.15

    @kelvin.setter
    def kelvin(self, value):
        self.celsius = value - 273.15

In [7]:
t = Temperature(100)
check("fahrenheit", lambda: t.fahrenheit == 212)
check("kelvin get", lambda: round(t.kelvin, 2) == 373.15)
def _set_k():
    t.kelvin = 273.15
    return round(t.celsius, 2) == 0
check("kelvin set", _set_k)
check("validation", lambda: raises(ValueError, Temperature, -300))
check("fahrenheit is read-only", lambda: raises(AttributeError, setattr, t, "fahrenheit", 5))

✅ PASS - fahrenheit
✅ PASS - kelvin get
✅ PASS - kelvin set
✅ PASS - validation
✅ PASS - fahrenheit is read-only


### 🟡 Problem 4 — Bank account with inheritance
`Account(owner, balance=0)` with `deposit`, `withdraw` (raises `ValueError("insufficient funds")` if it would go below 0) and `__repr__` `Account('Asha', 100)`.
`SavingsAccount(Account)` adds `rate` and method `add_interest()` (balance *= 1 + rate).
`CurrentAccount(Account)` allows an `overdraft` limit: balance may go down to `-overdraft`.

<details><summary>💡 Hint</summary>Put the "minimum allowed balance" logic in a method/attribute the subclass can override.</details>

In [8]:
class Account:
    def __init__(self, owner, balance=0):
        self.owner, self.balance = owner, balance

    def _floor(self):
        return 0

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit must be positive")
        self.balance += amount

    def withdraw(self, amount):
        if self.balance - amount < self._floor():
            raise ValueError("insufficient funds")
        self.balance -= amount

    def __repr__(self):
        return f"{type(self).__name__}({self.owner!r}, {self.balance})"

class SavingsAccount(Account):
    def __init__(self, owner, balance=0, rate=0.05):
        super().__init__(owner, balance)
        self.rate = rate

    def add_interest(self):
        self.balance *= 1 + self.rate

class CurrentAccount(Account):
    def __init__(self, owner, balance=0, overdraft=100):
        super().__init__(owner, balance)
        self.overdraft = overdraft

    def _floor(self):
        return -self.overdraft

In [9]:
a = Account("Asha", 100)
a.deposit(50); a.withdraw(30)
check("deposit/withdraw", lambda: a.balance == 120)
check("insufficient funds", lambda: raises(ValueError, a.withdraw, 1000))
check("repr", lambda: repr(Account("Asha", 100)) == "Account('Asha', 100)")
s = SavingsAccount("Ravi", 1000, rate=0.10)
s.add_interest()
check("interest", lambda: round(s.balance, 2) == 1100)
check("savings is-an Account", lambda: isinstance(s, Account))
c = CurrentAccount("Tom", 50, overdraft=100)
c.withdraw(120)
check("overdraft allowed", lambda: c.balance == -70)
check("beyond overdraft rejected", lambda: raises(ValueError, c.withdraw, 100))

✅ PASS - deposit/withdraw
✅ PASS - insufficient funds
✅ PASS - repr
✅ PASS - interest
✅ PASS - savings is-an Account
✅ PASS - overdraft allowed
✅ PASS - beyond overdraft rejected


### 🟡 Problem 5 — Operator overloading: `Vector2D`
Implement `Vector2D(x, y)` with: `+`, `-`, `*` (scalar, also `2 * v`), `==`, `abs()` (length), `__repr__` `Vector2D(1, 2)`, unpacking (`x, y = v`), and `bool(v)` false for the zero vector.

In [10]:
class Vector2D:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __add__(self, o): return Vector2D(self.x + o.x, self.y + o.y)
    def __sub__(self, o): return Vector2D(self.x - o.x, self.y - o.y)
    def __mul__(self, k): return Vector2D(self.x * k, self.y * k)
    __rmul__ = __mul__
    def __eq__(self, o): return isinstance(o, Vector2D) and (self.x, self.y) == (o.x, o.y)
    def __abs__(self): return (self.x ** 2 + self.y ** 2) ** 0.5
    def __repr__(self): return f"Vector2D({self.x}, {self.y})"
    def __iter__(self): yield self.x; yield self.y
    def __bool__(self): return bool(self.x or self.y)

In [11]:
v, w = Vector2D(3, 4), Vector2D(1, 2)
check("add", lambda: v + w == Vector2D(4, 6))
check("sub", lambda: v - w == Vector2D(2, 2))
check("scalar mult both sides", lambda: v * 2 == Vector2D(6, 8) and 2 * v == Vector2D(6, 8))
check("abs", lambda: abs(v) == 5.0)
check("repr", lambda: repr(w) == "Vector2D(1, 2)")
def _unpack():
    x, y = v
    return (x, y) == (3, 4)
check("unpacking", _unpack)
check("bool", lambda: bool(v) and not bool(Vector2D(0, 0)))

✅ PASS - add
✅ PASS - sub
✅ PASS - scalar mult both sides
✅ PASS - abs
✅ PASS - repr
✅ PASS - unpacking
✅ PASS - bool


### 🟡 Problem 6 — Polymorphism with an abstract base class
Create abstract `Shape` (ABC) with abstract `area()` and concrete `describe()` returning `"<ClassName> area=<area rounded to 2>"`. Subclasses `Circle(r)`, `Triangle(base, height)`, `Square(side)`. Shapes must not be instantiable if `area` isn't implemented. Then `total_area(shapes)`.

In [12]:
from abc import ABC, abstractmethod
import math

class Shape(ABC):
    @abstractmethod
    def area(self): ...

    def describe(self):
        return f"{type(self).__name__} area={round(self.area(), 2)}"

class Circle(Shape):
    def __init__(self, r): self.r = r
    def area(self): return math.pi * self.r ** 2

class Triangle(Shape):
    def __init__(self, base, height): self.base, self.height = base, height
    def area(self): return 0.5 * self.base * self.height

class Square(Shape):
    def __init__(self, side): self.side = side
    def area(self): return self.side ** 2

def total_area(shapes):
    return round(sum(s.area() for s in shapes), 2)

In [13]:
shapes = [Circle(1), Triangle(4, 3), Square(2)]
check("describe", lambda: Square(3).describe() == "Square area=9")
check("circle describe", lambda: Circle(1).describe() == "Circle area=3.14")
check("total_area", lambda: total_area(shapes) == 13.14)
check("Shape is abstract", lambda: raises(TypeError, Shape))
def _incomplete():
    class Bad(Shape): pass
    return raises(TypeError, Bad)
check("subclass without area can't be created", _incomplete)

✅ PASS - describe
✅ PASS - circle describe
✅ PASS - total_area
✅ PASS - Shape is abstract
✅ PASS - subclass without area can't be created


### 🟡 Problem 7 — Class methods & static methods
`Date2` stores `day, month, year`. Add:
- `@classmethod from_string("25-12-2024")` alternative constructor
- `@staticmethod is_valid(day, month, year)` (months 1–12, days 1–31 — keep it simple)
- `__str__` → `"25/12/2024"` (zero-padded day and month)

In [14]:
class Date2:
    def __init__(self, day, month, year):
        if not Date2.is_valid(day, month, year):
            raise ValueError("invalid date")
        self.day, self.month, self.year = day, month, year

    @classmethod
    def from_string(cls, text):
        d, m, y = map(int, text.split("-"))
        return cls(d, m, y)

    @staticmethod
    def is_valid(day, month, year):
        return 1 <= month <= 12 and 1 <= day <= 31 and year > 0

    def __str__(self):
        return f"{self.day:02d}/{self.month:02d}/{self.year}"

In [15]:
check("from_string", lambda: str(Date2.from_string("5-3-2024")) == "05/03/2024")
check("is_valid static", lambda: Date2.is_valid(31, 12, 2000) and not Date2.is_valid(32, 1, 2000))
check("invalid constructor raises", lambda: raises(ValueError, Date2, 1, 13, 2000))
check("attributes", lambda: (lambda d: (d.day, d.month, d.year))(Date2.from_string("25-12-2024")) == (25, 12, 2024))

✅ PASS - from_string
✅ PASS - is_valid static
✅ PASS - invalid constructor raises
✅ PASS - attributes


### 🟡 Problem 8 — Stack class with dunder methods
Implement `Stack` with `push`, `pop` (raises `IndexError("empty")` when empty), `peek`, `__len__`, `__bool__`-via-len, `__iter__` (top to bottom), `__contains__`, and `__repr__` like `Stack([1, 2, 3])` (bottom → top).

In [16]:
class Stack:
    def __init__(self):
        self._items = []

    def push(self, x): self._items.append(x)

    def pop(self):
        if not self._items:
            raise IndexError("empty")
        return self._items.pop()

    def peek(self):
        if not self._items:
            raise IndexError("empty")
        return self._items[-1]

    def __len__(self): return len(self._items)
    def __iter__(self): return iter(reversed(self._items))
    def __contains__(self, x): return x in self._items
    def __repr__(self): return f"Stack({self._items})"

In [17]:
s = Stack()
for x in (1, 2, 3): s.push(x)
check("repr", lambda: repr(s) == "Stack([1, 2, 3])")
check("len / peek", lambda: len(s) == 3 and s.peek() == 3)
check("iter top->bottom", lambda: list(s) == [3, 2, 1])
check("contains", lambda: 2 in s and 9 not in s)
check("pop order", lambda: [s.pop(), s.pop(), s.pop()] == [3, 2, 1])
check("empty is falsy", lambda: not s)
check("pop from empty", lambda: raises(IndexError, s.pop))

✅ PASS - repr
✅ PASS - len / peek
✅ PASS - iter top->bottom
✅ PASS - contains
✅ PASS - pop order
✅ PASS - empty is falsy
✅ PASS - pop from empty


### 🔴 Problem 9 — Composition: `Library`
`Book(title, author)` has `available` (bool). `Library` *has* books (a dict by title). Methods: `add(book)`, `checkout(title)` (raises `KeyError` if unknown, `ValueError("already checked out")` if unavailable), `give_back(title)`, `available_titles()` (sorted).

In [18]:
class Book:
    def __init__(self, title, author):
        self.title, self.author, self.available = title, author, True

class Library:
    def __init__(self):
        self.books = {}

    def add(self, book):
        self.books[book.title] = book

    def checkout(self, title):
        book = self.books[title]                 # KeyError if unknown
        if not book.available:
            raise ValueError("already checked out")
        book.available = False

    def give_back(self, title):
        self.books[title].available = True

    def available_titles(self):
        return sorted(t for t, b in self.books.items() if b.available)

In [19]:
lib = Library()
lib.add(Book("Dune", "Herbert")); lib.add(Book("Emma", "Austen")); lib.add(Book("Ulysses", "Joyce"))
lib.checkout("Dune")
check("available after checkout", lambda: lib.available_titles() == ["Emma", "Ulysses"])
check("double checkout", lambda: raises(ValueError, lib.checkout, "Dune"))
check("unknown title", lambda: raises(KeyError, lib.checkout, "Nope"))
def _back():
    lib.give_back("Dune"); return lib.available_titles() == ["Dune", "Emma", "Ulysses"]
check("give_back", _back)

✅ PASS - available after checkout
✅ PASS - double checkout
✅ PASS - unknown title
✅ PASS - give_back


### 🔴 Problem 10 — Multiple inheritance & MRO
`Service(Logger, Timer)` inherits from two classes that both inherit from `Base`. Implement a **cooperative** `run()` in `Timer`, `Logger` and `Service`: each one calls `super().run()` **first**, then appends its own entry `"<ClassName>:<name>"` to the returned list.

So `Service("db").run()` must return `["Timer:db", "Logger:db", "Service:db"]`.

Why that order? `Service.__mro__` is `Service → Logger → Timer → Base → object`. `super()` walks *down* that chain until `Base.run()` returns `[]`; then each level appends on the way back *up*, so the class **furthest down the MRO appends first**. Print `Service.__mro__` to see it.

In [20]:
class Base:
    def __init__(self, name):
        self.name = name
    def run(self):
        return []

class Timer(Base):
    def run(self):
        return super().run() + [f"Timer:{self.name}"]

class Logger(Base):
    def run(self):
        return super().run() + [f"Logger:{self.name}"]

class Service(Logger, Timer):
    def run(self):
        return super().run() + [f"Service:{self.name}"]

In [21]:
check("run order follows MRO", lambda: Service("db").run() == ["Timer:db", "Logger:db", "Service:db"])
check("MRO", lambda: [c.__name__ for c in Service.__mro__] == ["Service", "Logger", "Timer", "Base", "object"])

✅ PASS - run order follows MRO
✅ PASS - MRO


### 🔴 Problem 11 — Dataclass & sorting
Use `@dataclass(order=True)` to define `Employee(name, salary)` where objects sort by **salary** (put `salary` first, and use `field(compare=False)` for `name`). Then write `top_earners(employees, n)` returning the names of the `n` highest-paid, highest first.

In [22]:
from dataclasses import dataclass, field

@dataclass(order=True)
class Employee:
    salary: float
    name: str = field(compare=False, default="")

def top_earners(employees, n):
    return [e.name for e in sorted(employees, reverse=True)[:n]]

In [23]:
staff = [Employee(50000, "Asha"), Employee(72000, "Kiran"), Employee(61000, "Meena")]
check("ordering by salary", lambda: Employee(1, "z") < Employee(2, "a"))
check("top 2", lambda: top_earners(staff, 2) == ["Kiran", "Meena"])
check("name is ignored in comparisons", lambda: Employee(5, "x") == Employee(5, "y"))

✅ PASS - ordering by salary
✅ PASS - top 2
✅ PASS - name is ignored in comparisons


### 🔴 Problem 12 — Design: a tiny event system (Observer pattern)
Class `EventEmitter` with `on(event, callback)`, `off(event, callback)` and `emit(event, *args)`, which calls every callback registered for that event in registration order and returns the list of their return values. Emitting an event with no listeners returns `[]`.

In [24]:
class EventEmitter:
    def __init__(self):
        self._listeners = {}

    def on(self, event, callback):
        self._listeners.setdefault(event, []).append(callback)

    def off(self, event, callback):
        if callback in self._listeners.get(event, []):
            self._listeners[event].remove(callback)

    def emit(self, event, *args):
        return [cb(*args) for cb in list(self._listeners.get(event, []))]

In [25]:
em = EventEmitter()
def double(x): return x * 2
def square(x): return x * x
em.on("num", double); em.on("num", square)
check("emit calls in order", lambda: em.emit("num", 5) == [10, 25])
check("no listeners", lambda: em.emit("other") == [])
def _off():
    em.off("num", double); return em.emit("num", 3) == [9]
check("off removes", _off)
check("off unknown is harmless", lambda: em.off("zzz", double) is None)

✅ PASS - emit calls in order
✅ PASS - no listeners
✅ PASS - off removes
✅ PASS - off unknown is harmless


---
## 🏁 Next
≥ 90 %? Go to `07_Algorithms_DataStructures_challenges.ipynb`. **Bonus:** add `__lt__` to `Vector2D` (by length) and use `sorted()`; implement `__getitem__` so `v[0]` works.